In [1]:
print("Ireland Energy Fabric notebook connected successfully")

StatementMeta(, 46b2d652-c6e9-4ca9-a39b-7347afdbfba4, 3, Finished, Available, Finished, False)

Ireland Energy Fabric notebook connected successfully


In [2]:
import os

base_path = "/lakehouse/default/Files/bronze"

print("Bronze folders:")
print(os.listdir(base_path))

StatementMeta(, 46b2d652-c6e9-4ca9-a39b-7347afdbfba4, 4, Finished, Available, Finished, False)

Bronze folders:
['epa', 'seai']


In [3]:
epa_path = "/lakehouse/default/Files/bronze/epa/2026-07"
seai_path = "/lakehouse/default/Files/bronze/seai/2026-09"

print("EPA files:")
print(os.listdir(epa_path))

print("\nSEAI files:")
print(os.listdir(seai_path))

StatementMeta(, 46b2d652-c6e9-4ca9-a39b-7347afdbfba4, 5, Finished, Available, Finished, False)

EPA files:
['GHG_Provisional_Emissions_1990-2025_web.xlsx']

SEAI files:
['National-Energy-Balance.xlsx']


In [4]:
import hashlib

epa_file = epa_path + "/GHG_Provisional_Emissions_1990-2025_web.xlsx"
seai_file = seai_path + "/National-Energy-Balance.xlsx"

def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            data = file.read(1024 * 1024)

            if not data:
                break

            sha256.update(data)

    return sha256.hexdigest()

epa_hash = calculate_sha256(epa_file)
seai_hash = calculate_sha256(seai_file)

print("EPA SHA-256:")
print(epa_hash)

print("\nSEAI SHA-256:")
print(seai_hash)

StatementMeta(, 46b2d652-c6e9-4ca9-a39b-7347afdbfba4, 6, Finished, Available, Finished, False)

EPA SHA-256:
a8dd1c32b9e7cb1f0ee57ae54ee08a9325892519bf6b6acf19a6e3c430cafe16

SEAI SHA-256:
bfb5ea11919ed4a31487c387fcfa9ebbd519561888010df79021c2db2dc61436


In [5]:
import uuid
from datetime import datetime

run_id = str(uuid.uuid4())
ingestion_time = datetime.now()

print("Run ID:")
print(run_id)

print("\nIngestion Time:")
print(ingestion_time)

StatementMeta(, 46b2d652-c6e9-4ca9-a39b-7347afdbfba4, 7, Finished, Available, Finished, False)

Run ID:
960c4a6b-d53b-45e9-9a57-bd8ad36e1322

Ingestion Time:
2026-10-08 10:24:01.579880


In [6]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

schema = StructType([
    StructField("source_name", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("source_url", StringType(), True),
    StructField("release_date", StringType(), False),
    StructField("downloaded_at", TimestampType(), False),
    StructField("file_hash", StringType(), False),
    StructField("run_id", StringType(), False),
    StructField("status", StringType(), False)
])

manifest_data = [
    (
        "EPA",
        "GHG_Provisional_Emissions_1990-2025_web.xlsx",
        "https://www.epa.ie/",
        "2026-07",
        ingestion_time,
        epa_hash,
        run_id,
        "SUCCESS"
    ),
    (
        "SEAI",
        "National-Energy-Balance.xlsx",
        "https://www.seai.ie/",
        "2026-09",
        ingestion_time,
        seai_hash,
        run_id,
        "SUCCESS"
    )
]

manifest_df = spark.createDataFrame(manifest_data, schema)

display(manifest_df)

StatementMeta(, 46b2d652-c6e9-4ca9-a39b-7347afdbfba4, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fedf7325-c61e-4341-962d-4342c91b547a)

In [7]:
manifest_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ingestion_manifest")

print("ingestion_manifest table created successfully")

StatementMeta(, 46b2d652-c6e9-4ca9-a39b-7347afdbfba4, 9, Finished, Available, Finished, False)

ingestion_manifest table created successfully


In [8]:
display(spark.table("ingestion_manifest"))

StatementMeta(, 46b2d652-c6e9-4ca9-a39b-7347afdbfba4, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e0fb26f5-0e9d-4bc4-96b1-c8779da5a98b)